In [0]:
%sql

INSERT OVERWRITE proyecto_final.silver.escuelas (
  id_establecimiento,
  nombre_funcional,
  nombre_generico,
  nombre_establecimiento,
  codigo_cui,
  codigo_cue,
  codigo_anexo,
  tipo_gestion,
  nivel_educativo,
  tipo_escuela,
  dependencia,
  direccion,
  barrio,
  comuna,
  fuente_datos,
  coordenadas,
  tipo_entidad,
  longitud,
  latitud
)
WITH datos_limpios AS (
  SELECT 
    id_establecimiento,
    LOWER(REPLACE(TRIM(nombre_funcional), '"', '')) AS nombre_funcional,
    LOWER(REPLACE(TRIM(nombre_generico), '"', '')) AS nombre_generico,
    LOWER(REPLACE(TRIM(nombre_establecimiento), '"', '')) AS nombre_establecimiento,
    CAST(codigo_cui AS INT) AS codigo_cui,
    CAST(codigo_cue AS INT) AS codigo_cue,
    CAST(codigo_anexo AS INT) AS codigo_anexo,
    LOWER(REPLACE(TRIM(tipo_gestion), '"', '')) AS tipo_gestion,
    LOWER(REPLACE(TRIM(nivel_educativo), '"', '')) AS nivel_educativo,
    LOWER(REPLACE(TRIM(tipo_escuela), '"', '')) AS tipo_escuela,
    LOWER(REPLACE(TRIM(dependencia), '"', '')) AS dependencia,
    LOWER(REPLACE(TRIM(direccion), '"', '')) AS direccion,
    LOWER(REPLACE(TRIM(barrio), '"', '')) AS barrio,
    comuna,
    LOWER(REPLACE(TRIM(fuente_datos), '"', '')) AS fuente_datos,
    'escuela' AS tipo_entidad,
    geometry,
    CAST(st_x(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS DOUBLE) AS longitud,
    CAST(st_y(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS DOUBLE) AS latitud
  FROM proyecto_final.raw.escuelas_bronze
  WHERE nombre_establecimiento IS NOT NULL 
    AND geometry IS NOT NULL
    AND nivel_educativo RLIKE '(?i)inicial|primario|secundario'
    AND nivel_educativo NOT RLIKE '(?i)profesional|laboral|talleres|superior|artística'
),
datos_deduplicados AS (
  SELECT *,
    CONCAT('POINT (', longitud, ' ', latitud, ')') AS coordenadas,
    ROW_NUMBER() OVER (PARTITION BY nombre_establecimiento, geometry ORDER BY id_establecimiento) AS rn
    FROM datos_limpios
)
SELECT 
    id_establecimiento,
    nombre_funcional,
    nombre_generico,
    nombre_establecimiento,
    codigo_cui,
    codigo_cue,
    codigo_anexo,
    tipo_gestion,
    nivel_educativo,
    tipo_escuela,
    dependencia,
    direccion,
    CASE WHEN barrio= 'boca' THEN 'la boca' ELSE barrio END AS barrio,
    comuna,
    fuente_datos,
    coordenadas,
    tipo_entidad,
    longitud,
    latitud
FROM datos_deduplicados
WHERE rn = 1;